In [35]:
import pandas as pd
import cbbpy.mens_scraper as s
import boto3
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from clean_names import normalize_team_name
from scipy.stats import norm


In [54]:
today = datetime.now(ZoneInfo("America/New_York"))
todays_games_ids = s.get_game_ids(date=today)

games = []

for i, game_id in enumerate(todays_games_ids, start=1):
    try:
        game, _, _ = s.get_game(game_id, box = False, pbp = False)
        games.append(game)
        print(f'Successfully recieved {game_id}! - {i} / {len(todays_games_ids)}')
    except Exception as e:
        print(f"Failed to retrieve {game_id}: {e}")

schedule = pd.concat(games, ignore_index=True)

Successfully recieved 401851485! - 1 / 46
Successfully recieved 401829290! - 2 / 46
Successfully recieved 401851698! - 3 / 46
Successfully recieved 401851454! - 4 / 46
Successfully recieved 401851620! - 5 / 46
Successfully recieved 401851546! - 6 / 46
Successfully recieved 401851545! - 7 / 46
Successfully recieved 401851544! - 8 / 46
Successfully recieved 401851543! - 9 / 46
Successfully recieved 401851526! - 10 / 46
Successfully recieved 401851486! - 11 / 46
Successfully recieved 401828732! - 12 / 46
Successfully recieved 401828259! - 13 / 46
Successfully recieved 401828257! - 14 / 46
Successfully recieved 401823056! - 15 / 46
Successfully recieved 401804962! - 16 / 46
Successfully recieved 401804958! - 17 / 46
Successfully recieved 401804957! - 18 / 46
Successfully recieved 401827126! - 19 / 46
Successfully recieved 401825560! - 20 / 46
Successfully recieved 401825561! - 21 / 46
Successfully recieved 401828670! - 22 / 46
Successfully recieved 401827106! - 23 / 46
Successfully recieve

In [55]:
cols = ['home_team', 'away_team', 'is_neutral', 'game_time', 'tv_network']
schedule = schedule[cols]
schedule['home_team'] = schedule['home_team'].apply(normalize_team_name)
schedule['away_team'] = schedule['away_team'].apply(normalize_team_name)

In [38]:
current_rankings = pd.read_csv('data/current_rankings.csv')
current_rankings = current_rankings[current_rankings['Date'] == '2026-03-01']

team_rating_map = current_rankings.set_index('Team')['Total']
schedule['home_team_rating'] = schedule['home_team'].map(team_rating_map)
schedule['away_team_rating'] = schedule['away_team'].map(team_rating_map)

In [45]:
def home_team_spread(home, away, neutral):
    spread = (home - away) * 0.679
    spread += (~neutral) * 2.9
    return spread

def home_team_wp(home_spread):
    home_wp = 1 - norm.cdf(0, loc=home_spread, scale=10)
    away_wp = 1 - home_wp
    return home_wp

In [53]:
schedule['home_team_spread'] = home_team_spread(schedule['home_team_rating'], schedule['away_team_rating'], schedule['is_neutral'])
schedule['home_team_wp'] = home_team_wp(schedule['home_team_spread'])
schedule['away_team_wp'] = 1 - schedule['home_team_wp']
schedule['home_team_spread'] = schedule['home_team_spread'] * -1

schedule[['home_team_wp', 'away_team_wp']] = (schedule[['home_team_wp', 'away_team_wp']] * 100).round(2)
schedule['home_team_spread'] = schedule['home_team_spread'].round(1)